# 🧠 Day 3A — CoT Warm-Up: Upgrading Your Day 2 ReAct Agent
**Agentic AI for Tax Technologists · 12-Day Program**

---

## 📖 Story So Far

On **Day 2** you built a `TaxFAQAgent` with the ReAct pattern:

```
PERCEIVE  →  REASON (implicit)  →  ACT (tool call)  →  OBSERVE  →  ANSWER
```

**The observation from Day 2:** The REASON step was *inconsistent*. Sometimes the agent showed beautiful step-by-step logic. Other times it jumped straight to an answer. You couldn't rely on the reasoning trace — which matters when a tax partner asks *"how did you get there?"*

**Today (Day 3A):** We upgrade the REASON step using **Chain-of-Thought (CoT)** prompting.
Same ReAct loop. Better, controllable thinking.

---

## 🎯 Learning Objectives
1. Understand how CoT differs from vanilla prompting
2. Add a CoT instruction to the Day 2 system prompt
3. Observe and compare reasoning depth across 5 GST questions
4. Measure qualitative improvement before formalising it in Notebook 3B

---
## ⏱️ Time: 30 minutes

---
## ⚙️ Step 1: Install & Import

In [ ]:
%pip install openai --quiet

In [ ]:
import os
import json
import datetime
from openai import AzureOpenAI
from getpass import getpass

print("✅ Imports OK")

---
## 🔑 Step 2: Configure Azure OpenAI Client

In [ ]:
AZURE_OPENAI_ENDPOINT = input("Endpoint (e.g. https://xxx.openai.azure.com/): ").strip()
AZURE_OPENAI_API_KEY  = getpass("API Key: ")
AZURE_DEPLOYMENT_NAME = input("Deployment name (e.g. gpt-4o): ").strip()

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = "2024-08-01-preview",
    timeout        = 120.0,
    max_retries    = 3
)
print("✅ Azure OpenAI client initialised")

---
## 🔁 Step 3: Day 2 Baseline — The Original System Prompt

This is what your Day 2 `TaxFAQAgent` used. We keep it unchanged to have a fair baseline.

In [ ]:
# ── Day 2 system prompt (BASELINE — do not change this cell) ──────────────────
DAY2_SYSTEM_PROMPT = (
    "You are TaxBot, an expert AI assistant for tax professionals at a Big4 firm.\n"
    "Rules:\n"
    "1. Always call search_tax_knowledge before answering any factual tax question.\n"
    "2. Use calculate_tax for any rupee computation.\n"
    "3. Use get_filing_deadline for any due date or deadline questions.\n"
    "4. Never fabricate statutory section numbers or rates.\n"
    "5. Qualify answers: as per current regulations, please verify for latest updates."
)

print("Day 2 system prompt loaded.")
print("-" * 60)
print(DAY2_SYSTEM_PROMPT)

In [ ]:
# ── Reusable caller function — same pattern as Day 2 ──────────────────────────
def ask_agent(system_prompt, question, label=""):
    """Call Azure OpenAI with a system prompt and return the response text."""
    response = client.chat.completions.create(
        model       = AZURE_DEPLOYMENT_NAME,
        messages    = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question},
        ],
        temperature = 0.0,
        max_tokens  = 700,
    )
    answer = response.choices[0].message.content
    if label:
        print(f"\n{'='*65}")
        print(f"  [{label}]")
        print(f"{'='*65}")
        print(f"Q: {question}")
        print(f"\nA:\n{answer}")
        print()
    return answer

print("✅ Helper function ready")

---
## 🧪 Step 4: Run the Baseline — 5 GST Questions

Read each answer carefully. Ask yourself:
- Did the agent show its reasoning steps, or did it jump straight to an answer?
- Would you be comfortable showing this reasoning trail to a tax partner?

In [ ]:
TEST_QUESTIONS = [
    "What GST rate applies to a software subscription sold by an Indian company to a US enterprise?",
    "A freelance graphic designer in Pune earns Rs 15 lakh per year. Do they need to register for GST?",
    "What is the difference between CGST, SGST, and IGST? When does each apply?",
    "A manufacturing company imports raw materials worth USD 50,000. What customs duty and IGST applies?",
    "Can a composition scheme dealer issue a tax invoice and collect GST from customers?",
]

baseline_answers = []
for i, q in enumerate(TEST_QUESTIONS):
    ans = ask_agent(DAY2_SYSTEM_PROMPT, q, label=f"BASELINE Q{i+1}")
    baseline_answers.append(ans)

---
## ✨ Step 5: Upgrade — Add Chain-of-Thought to the System Prompt

We try two CoT techniques:

| Technique | What changes | When to use |
|-----------|-------------|-------------|
| **CoT-A** (zero-shot CoT) | Append `"think step by step"` | Quick upgrade, any existing prompt |
| **CoT-B** (structured CoT) | Enforce explicit 4-step format | Production — need consistent, auditable trace |

In [ ]:
# ── Technique A: Zero-Shot CoT — one instruction appended to the Day 2 prompt ──
COT_ZEROSHOT_SYSTEM = (
    DAY2_SYSTEM_PROMPT
    + "\n\n"
    + "REASONING INSTRUCTION:\n"
    + "Before giving any answer, always think step by step.\n"
    + "Show your reasoning explicitly. Do not jump to conclusions."
)

print("📝 Technique A — Zero-Shot CoT prompt:")
print("-" * 60)
print(COT_ZEROSHOT_SYSTEM)

In [ ]:
# ── Technique B: Structured CoT — explicit 4-step reasoning format ─────────────
COT_STRUCTURED_SYSTEM = (
    "You are TaxBot, a senior GST consultant at a Big4 firm.\n\n"
    "For EVERY question you answer, follow this exact 4-step reasoning format:\n\n"
    "STEP 1 — CLASSIFY: Identify the type of transaction or scenario.\n"
    "STEP 2 — LOCATE THE RULE: State the applicable section / notification / circular.\n"
    "STEP 3 — APPLY THE RULE: Work through the logic for this specific case.\n"
    "STEP 4 — FINAL ANSWER: State the answer clearly and concisely.\n\n"
    "Never skip a step. Never give FINAL ANSWER before completing steps 1–3.\n"
    "If you are uncertain, say so explicitly in the relevant step.\n"
    "Never fabricate statutory section numbers or rates."
)

print("📝 Technique B — Structured CoT prompt:")
print("-" * 60)
print(COT_STRUCTURED_SYSTEM)

---
## 🔬 Step 6: Run Both CoT Variants on the Same 5 Questions

In [ ]:
# Run Technique A — Zero-Shot CoT
cot_a_answers = []
for i, q in enumerate(TEST_QUESTIONS):
    ans = ask_agent(COT_ZEROSHOT_SYSTEM, q, label=f"CoT-A (zero-shot)   Q{i+1}")
    cot_a_answers.append(ans)

In [ ]:
# Run Technique B — Structured CoT
cot_b_answers = []
for i, q in enumerate(TEST_QUESTIONS):
    ans = ask_agent(COT_STRUCTURED_SYSTEM, q, label=f"CoT-B (structured)  Q{i+1}")
    cot_b_answers.append(ans)

---
## 📊 Step 7: Score & Compare

After reading the outputs above, fill in your scores in the cell below.

| Dimension | 0 | 1 | 2 |
|---|---|---|---|
| **Reasoning depth** | No reasoning shown | Some reasoning | Full explicit steps |
| **Step structure** | Free-form | Partial structure | All steps labelled |
| **Answer correctness** | Wrong | Partially correct | Fully correct |

In [ ]:
# ── TODO: Update these scores after reading the outputs ───────────────────────
# Format per question: [reasoning_depth, step_structure, correctness]  (each 0/1/2)

scores = {
    "Baseline (Day 2)": [
        [0, 0, 1],  # Q1 — edit after reading
        [0, 0, 1],  # Q2
        [0, 0, 1],  # Q3
        [0, 0, 1],  # Q4
        [0, 0, 1],  # Q5
    ],
    "CoT-A (zero-shot)": [
        [1, 0, 1],
        [1, 0, 1],
        [1, 0, 1],
        [1, 0, 1],
        [1, 0, 1],
    ],
    "CoT-B (structured)": [
        [2, 2, 2],
        [2, 2, 2],
        [2, 2, 2],
        [2, 2, 2],
        [2, 2, 2],
    ],
}

print(f"{'Approach':<22} {'Reasoning':>10} {'Structure':>10} {'Correctness':>12} {'TOTAL /30':>10}")
print("-" * 67)
for name, qs in scores.items():
    r = sum(q[0] for q in qs)
    s = sum(q[1] for q in qs)
    c = sum(q[2] for q in qs)
    print(f"{name:<22} {r:>8}/10 {s:>8}/10 {c:>10}/10 {r+s+c:>8}/30")

print()
print("💡 Edit the scores above with your actual observations, then re-run this cell.")

---
## 🔎 Step 8: Side-by-Side on One Question

In [ ]:
# Change Q_IDX (0–4) to inspect any question
Q_IDX = 0

print("\n" + "#" * 70)
print(f"QUESTION {Q_IDX+1}: {TEST_QUESTIONS[Q_IDX]}")
print("#" * 70)

for label, answers in [
    ("BASELINE (Day 2)",    baseline_answers),
    ("CoT-A  (zero-shot)",  cot_a_answers),
    ("CoT-B  (structured)", cot_b_answers),
]:
    print(f"\n{'─' * 60}")
    print(f"  {label}")
    print(f"{'─' * 60}")
    print(answers[Q_IDX])

print("\n" + "#" * 70)
print("REFLECTION: Which answer would you trust to show a tax partner? Why?")
print("#" * 70)

---
## 💾 Step 9: Save Results to File

In [ ]:
output = {
    "notebook":    "Day3A_CoT_Warmup",
    "exported_at": datetime.datetime.now().isoformat(),
    "questions":   TEST_QUESTIONS,
    "baseline":    baseline_answers,
    "cot_a":       cot_a_answers,
    "cot_b":       cot_b_answers,
}

with open("day3a_results.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("✅ Results saved to day3a_results.json")

---
## 🎓 Key Insight

```
Day 2 ReAct loop:
  PERCEIVE  →  REASON (uncontrolled)  →  ACT  →  OBSERVE

Day 3 CoT-enhanced loop:
  PERCEIVE  →  REASON (4-step structured)  →  ACT  →  OBSERVE
```

The loop **didn't change** — we just made the REASON step explicit.

**CoT-B (structured)** is strictly better because:
- The reasoning is *always* there — the model cannot give a naked answer
- The steps are labelled — partners and auditors can follow the logic
- The structure is predictable — easier to parse in Notebook 3B

---
## ➡️ Next: Notebook 3B — Few-Shot CoT + JSON Output Parsing

We write 3 few-shot examples so the model *always* returns a structured JSON object — not just well-reasoned prose.